# Context Managers

In the previous notebook, we used `with` to open and automatically close files.

But `with` is not specific to files.

Python can use the same pattern whenever something needs to be prepared before a block and cleaned up afterward.

How does Python know what to do when entering and leaving a `with` block?

## The `with` Statement

A `with` statement defines a block of code where a context is managed automatically.

In [1]:
with open("example.txt", "w") as file:
    file.write("Hello, Python!")

The code inside the `with` block runs while the context is active.

When the block ends, Python performs the context manager's exit operation automatically.

## Context Managers

A context manager is an object that defines what should happen when entering and leaving a `with` block.

It implements two special methods:

- `__enter__()` — called when entering the block
- `__exit__()` — called when leaving the block

In [23]:
class Example:

    def __exit__(self):
        print("Leaving the context")

with Example():
    print("Inside the context")

TypeError: 'Example' object does not support the context manager protocol

In [22]:
class Example:
    def __enter__(self):
        print("Entering the context")

with Example():
    print("Inside the context")

TypeError: 'Example' object does not support the context manager protocol (missed __exit__ method)

In [7]:
class Example:
    def __enter__(self):
        print("Entering the context")

    def __exit__(self):
        print("Leaving the context")

with Example():
    print("Inside the context")

Entering the context
Inside the context


TypeError: Example.__exit__() takes 1 positional argument but 4 were given

The class above is not yet a valid context manager because `__exit__()` must accept information about an eventual exception.

In [8]:
class Example:
    def __enter__(self):
        print("Entering the context")

    def __exit__(self, exception_type, exception_value, traceback):
        print("Leaving the context")

with Example():
    print("Inside the context")

Entering the context
Inside the context
Leaving the context


The execution order is:

`__enter__()` → code inside the block → `__exit__()`

In [9]:
class Example:
    def __enter__(self):
        print("1. Enter")

    def __exit__(self, exception_type, exception_value, traceback):
        print("3. Exit")

with Example():
    print("2. Inside")

1. Enter
2. Inside
3. Exit


## Returning a Value from `__enter__()`

The value returned by `__enter__()` can be assigned using `as`.

In [12]:
class Example:
    def __enter__(self):
        return "Context value"

    def __exit__(self, exception_type, exception_value, traceback):
        pass

with Example() as value:
    print(value)

Context value


The value returned by `__enter__()` becomes the value assigned to `as`.

## `__exit__()` and Exceptions

`__exit__()` is also called when an exception occurs inside the `with` block.

In [14]:
class Example:
    def __enter__(self):
        print("Enter")

    def __exit__(self, exception_type, exception_value, traceback):
        print("Exit")
        print(exception_type)
        print(exception_value)
        print(traceback)

with Example():
    print("Inside")
    raise ValueError("Something went wrong")

Enter
Inside
Exit
<class 'ValueError'>
Something went wrong


ValueError: Something went wrong

The exception information is passed to `__exit__()`.

The three arguments describe:

- the exception type
- the exception value
- the traceback

## Why Context Managers Matter

A context manager centralizes setup and cleanup.

Instead of remembering to perform cleanup manually, the `with` statement guarantees that the exit operation is called when the block finishes, including when an exception occurs.

In [18]:
class Resource:
    def __enter__(self):
        print("Resource acquired")
        return self

    def __exit__(self, exception_type, exception_value, traceback):
        print("Resource released")

with Resource() as resource:
    print(f"Using resource: {resource}")

Resource acquired
Using resource: <__main__.Resource object at 0x1076ae510>
Resource released


This pattern is useful whenever a resource or temporary state needs to be:

1. prepared before use
2. used inside a block
3. cleaned up afterward

In [19]:
class Connection:
    def __enter__(self):
        print("Connection opened")
        return self

    def __exit__(self, exception_type, exception_value, traceback):
        print("Connection closed")

with Connection() as connection:
    print(f"Working with the connection: {connection}")

Connection opened
Working with the connection: <__main__.Connection object at 0x1076aea50>
Connection closed


## The Context Manager and `with`

The `with` statement and a context manager are not the same thing.

`with` is Python syntax.

A context manager is an object used by `with`.

The context manager defines what happens when Python enters and leaves the `with` block.

In [21]:
with open("data/example.txt", "r") as file:
    print(file.read())

New content.
Additional content.


In this example:

`with` → Python syntax

the object returned by `open()` → context manager

The `with` statement uses the context manager to manage the block automatically.

## Summary

- A context manager controls what happens when entering and leaving a `with` block.
- `__enter__()` runs when the block starts.
- `__exit__()` runs when the block ends, including when an exception occurs.
- The value returned by `__enter__()` can be assigned with `as`.
- Context managers matter because they make setup and cleanup automatic and predictable.